# Luna et al. 2017 grid and seven ratios — left-first, reviewed landmarks


Reference: Luna, Aranda & Santos (2017), International Journal of Osteoarchaeology 27:898–911, doi:10.1002/oa.2604, pp.900–902, Figure 2.

Original study used left ilia. Here right-only images will be horizontally reflected automatically and recorded as a right-derived methodological extension, not an observed left bone. No bilateral averaging. If a left exists but is unusable, explicitly override to R and record the reason.

Automatic landmarks are PROPOSALS, not validated anatomy. Masks omit anatomical context; confirm against original photographs where needed. In particular gamma is the superior point of the ANTERIOR area, not necessarily the global silhouette extreme. Beta is the auricular edge angle, not necessarily the deepest convexity defect.

Figure 2 coordinate convention (image x increases right, y increases down): anterior up, superior left, inferior right. Vertical lines x_gamma, x_beta, x_alpha. Horizontal lines y_delta, y_beta, y_epsilon. A B C are top row, D E F middle, G H I bottom; E=beta. Alpha/gamma are generally NOT corners. This corrects the earlier swapped-axis implementation.

Requires masks already approximately vertically oriented. Editor offers residual rotation; no image stretching. Ratios are dimensionless; segment lengths are in working-image pixels. No sex classifier included. Sample <7 or >18 years is outside the original study age range. Predictions would not be validation without independent sex labels.

Keep this notebook beside consolidated_metadata.csv and folders masks/, f/, and bones-assets/. Photoshop references open first (matching explicit side where available); raw photos stay in the dropdown. References are visual only and are not automatically aligned or used to infer points. The authoritative metadata is not overwritten: a linked copy is written to consolidated_metadata_with_references.csv. Existing annotation files remain in luna_output/annotations/. Unsaved editor drafts are in memory only; save before restarting or rerunning the editor. No automatic loading of previous approved annotations is added, to avoid unintentionally unblinding repeat observations.


## Installation (once)
Uncomment the installation command if needed, run it, then restart the kernel. Run the configuration cell next to enable the interactive backend.

In [1]:
# %pip install -U numpy pandas Pillow scipy opencv-python matplotlib ipywidgets ipympl

In [2]:
from pathlib import Path
import re,json,hashlib,uuid
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display,clear_output

import ipympl
%matplotlib ipympl

import matplotlib
import sys

print("Python:", sys.executable)
print("ipympl:", ipympl.__version__)
print("Backend:", matplotlib.get_backend())

ROOT=Path.cwd()
MASK_DIR=ROOT/'masks'
OUTPUT=ROOT/'luna_output'
OBSERVER_ID='observer_1'
SESSION_ID='session_1'  # change for repeat observations
RIGHT_ALREADY_REFLECTED=False
BACKGROUND_TOLERANCE=20
SIDE_OVERRIDES={}  # e.g. {'8':'R'} only if left is unsuitable
SIDE_OVERRIDE_REASONS={}  # e.g. {'8':'left border damaged'}
for sub in ['draft_overlays','reviewed_overlays','annotations']:(OUTPUT/sub).mkdir(parents=True,exist_ok=True)
metadata=pd.read_csv(ROOT/'consolidated_metadata.csv',sep=None,engine='python',dtype=str,keep_default_na=False)
metadata['individual_id']=metadata.individual_id.str.strip().str.replace(',','.',regex=False)
assert metadata.individual_id.is_unique
pattern=re.compile(r'^(?:f)?(?P<id>\d+(?:[.,-]\d+)?)[_ ]?(?P<side>[LR])(?:[^A-Za-z].*)?$',re.I)
rows=[]
for p in sorted(MASK_DIR.rglob('*')):
    if p.suffix.lower() not in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}:continue
    m=pattern.match(p.stem)
    if m:rows.append({'individual_id':m['id'].replace('-','.').replace(',','.'),'side':m['side'].upper(),'path':str(p),'filename':p.name})
files=pd.DataFrame(rows)
assert len(files),'No masks found'
assert not files.duplicated(['individual_id','side']).any(),'Resolve duplicate individual-side files'
selected=[]
for ident,g in files.groupby('individual_id'):
    if ident not in set(metadata.individual_id):continue
    side=SIDE_OVERRIDES.get(ident,'L' if 'L' in set(g.side) else 'R')
    assert side in set(g.side),f'Override side unavailable for {ident}'
    if ident in SIDE_OVERRIDES:assert SIDE_OVERRIDE_REASONS.get(ident),'Provide override reason'
    r=g[g.side==side].iloc[0].to_dict()
    r['selection_reason']=SIDE_OVERRIDE_REASONS.get(ident,'left preferred' if side=='L' else 'right only; reflected')
    selected.append(r)
selected=pd.DataFrame(selected).sort_values('individual_id',key=lambda s:pd.to_numeric(s)).reset_index(drop=True)
selected.to_csv(OUTPUT/'selected_source_masks.csv',index=False)
display(selected.groupby('side').size().rename('selected_count'))


Python: /usr/local/bin/python
ipympl: 0.10.0
Backend: ipympl


side
L    55
R    13
Name: selected_count, dtype: int64

## Geometry and automated proposals
Alpha/gamma and delta/epsilon start as silhouette extrema in the Figure 2 orientation; beta is proposed from the deepest hull defect, with a flagged positional fallback. These heuristics do not establish anatomical homology. Preview each grid and correct each point before approval. Automatic batch results stay separate from reviewed results.

Before editing, right masks are reflected, cropped with padding, and optionally rotated. Border contact and discarded foreground are logged. Rotation is applied to the binary working mask; it cannot correct camera perspective.

In [3]:
def load_working(path,side,angle=0):
    with Image.open(path) as im:
        rgba=im.convert('RGBA')
        gray=np.asarray(Image.alpha_composite(Image.new('RGBA',rgba.size,'white'),rgba).convert('L'))
    binary=gray.astype(np.int16)<255-BACKGROUND_TOLERANCE
    if not binary.any():raise ValueError('No foreground')
    border=bool(binary[0].any() or binary[-1].any() or binary[:,0].any() or binary[:,-1].any())
    lab,n=ndimage.label(binary);counts=np.bincount(lab.ravel());counts[0]=0
    largest=lab==counts.argmax();discarded=float(1-largest.sum()/binary.sum())
    mask=ndimage.binary_fill_holes(largest)
    reflected=side=='R' and not RIGHT_ALREADY_REFLECTED
    if reflected:mask=np.fliplr(mask)
    y,x=np.where(mask);mask=mask[y.min():y.max()+1,x.min():x.max()+1]
    mask=np.pad(mask,max(10,int(.08*max(mask.shape))))
    if angle:mask=ndimage.rotate(mask.astype('uint8'),angle,reshape=True,order=0,cval=0)>0
    contours,_=cv2.findContours(mask.astype('uint8'),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    contour=max(contours,key=cv2.contourArea)
    return mask,contour,{'source_touches_border':border,'discarded_foreground_fraction':discarded,'reflected':reflected,'rotation_degrees':float(angle)}

def propose(contour):
    contour=np.asarray(contour,dtype=np.int32).reshape(-1,1,2)
    pts=contour.reshape(-1,2).astype(float)
    p={'alpha':pts[pts[:,0].argmax()].tolist(),'gamma':pts[pts[:,0].argmin()].tolist(),
       'delta':pts[pts[:,1].argmin()].tolist(),'epsilon':pts[pts[:,1].argmax()].tolist()}
    beta=None
    try:
        hull=cv2.convexHull(contour,returnPoints=False)
        defects=cv2.convexityDefects(contour,hull) if len(hull)>3 else None
        if defects is not None:
           # Handle either (N, 1, 4) or (N, 4) defect arrays.
            defect_rows = np.asarray(defects).reshape(-1, 4)

            if len(defect_rows) > 0:
                defect = defect_rows[np.argmax(defect_rows[:, 3])]

                if defect[3] > 0:
                    beta = pts[int(defect[2])].tolist()
    except cv2.error:pass
    fallback=beta is None
    if fallback:beta=[float(np.mean([p['gamma'][0],p['alpha'][0]])),float(np.mean([p['delta'][1],p['epsilon'][1]]))]
    p['beta']=beta
    return p,fallback

RATIO_PAIRS={'AC_CI':('AC','CI'),'FI_CF':('FI','CF'),'AB_BC':('AB','BC'),'HI_EH':('HI','EH'),'EF_CF':('EF','CF'),'DE_AD':('DE','AD'),'DE_EH':('DE','EH')}
def geometry(p):
    x0,x1,x2=p['gamma'][0],p['beta'][0],p['alpha'][0]
    y0,y1,y2=p['delta'][1],p['beta'][1],p['epsilon'][1]
    if not (x0<x1<x2 and y0<y1<y2):raise ValueError('Require x_gamma < x_beta < x_alpha and y_delta < y_beta < y_epsilon. Check anatomy/orientation, not just numerical order.')
    grid={letter:np.array([x,y],float) for letter,(x,y) in zip('ABCDEFGHI',[(x,y) for y in [y0,y1,y2] for x in [x0,x1,x2]])}
    lengths={s:float(np.linalg.norm(grid[s[1]]-grid[s[0]])) for s in {s for pair in RATIO_PAIRS.values() for s in pair}}
    ratios={name:lengths[a]/lengths[b] for name,(a,b) in RATIO_PAIRS.items()}
    return grid,lengths,ratios

def plot_grid(mask,p,title):
    fig,ax=plt.subplots(figsize=(7,8));ax.imshow(mask,cmap='gray_r')
    try:
        grid,lengths,ratios=geometry(p)
        for a,b in [('A','C'),('D','F'),('G','I'),('A','G'),('B','H'),('C','I')]:
            ax.plot([grid[a][0],grid[b][0]],[grid[a][1],grid[b][1]],color='tab:orange',lw=1)
        for letter,xy in grid.items():ax.annotate(letter,xy,xytext=(3,-8),textcoords='offset points',color='darkred')
    except ValueError as e:ax.text(.01,.01,str(e),transform=ax.transAxes,color='red',wrap=True)
    for label,xy in p.items():ax.plot(*xy,'o',ms=4,color='tab:blue');ax.annotate(label,xy,xytext=(5,8),textcoords='offset points',fontsize=8,color='tab:blue')
    ax.set_title(title);ax.set_aspect('equal');ax.axis('off');fig.tight_layout()
    return fig

# Exact arithmetic check of the grid mapping (not a simulated specimen).
test={'gamma':[0,1],'alpha':[10,2],'beta':[4,3],'delta':[4,0],'epsilon':[6,9]}
g,l,r=geometry(test)
assert np.allclose([r[k] for k in RATIO_PAIRS],[10/9,2,4/6,1,2,4/3,2/3])


In [4]:
drafts=[];working={};failures=[]
for r in selected.itertuples():
    try:
        mask,contour,info=load_working(r.path,r.side)
        points,fallback=propose(contour)
        working[r.individual_id]={'mask':mask,'points':points,'info':info,'row':r._asdict(),'fallback':fallback}
        record={'individual_id':r.individual_id,'source_side':r.side,'source_filename':r.filename,'status':'unreviewed_proposal',**info,'beta_positional_fallback':fallback}
        try:_,lengths,ratios=geometry(points);record.update(ratios)
        except ValueError as e:record['geometry_error']=str(e)
        drafts.append(record)
        with plt.ioff():
            fig=plot_grid(mask,points,f'{r.individual_id} | source {r.side} | UNREVIEWED')
        fig.savefig(OUTPUT/'draft_overlays'/f'{r.individual_id}.png',dpi=130);plt.close(fig)
    except Exception as e:
        print(f"Failed on skeleton {r.individual_id}: {e}")
        failures.append({'individual_id':r.individual_id,'error':f'{type(e).__name__}: {e}'})

pd.DataFrame(drafts).to_csv(
    OUTPUT / "automatic_proposals_UNREVIEWED.csv",
    index=False,
)

pd.DataFrame(
    failures,
    columns=["individual_id", "error"],
).to_csv(
    OUTPUT / "processing_failures.csv",
    index=False,
)

assert working, "No masks processed"

print(
    f"{len(working)} proposed grids. "
    "Review with the editor below; these are not approved measurements."
)

68 proposed grids. Review with the editor below; these are not approved measurements.


## Processing check
Do not approve missing or failed proposals. Inspect processing_failures.csv if any failures are reported.

In [5]:
print(f'Usable proposals: {len(working)}; failures: {len(failures)}')
display(pd.DataFrame(failures,columns=['individual_id','error']))

Usable proposals: 68; failures: 0


,individual_id,error


## Reference images and database links
Scans local files and writes a reusable reference manifest and linked metadata copy. Missing optional folders are reported without preventing grid review.

In [6]:
import io, html
from PIL import ImageOps

# Discover references locally; never assign anatomical side from a raw filename.
reference_rows=[]; unparsed=[]
asset_pattern=re.compile(r'^(?P<id>\d+(?:[.,-]\d+)?)(?P<side>[LR])?$',re.I)
raw_pattern=re.compile(r'^f(?P<id>\d+(?:[.,-]\d+)?)P.+$',re.I)
for folder,kind,pattern in [(ROOT/'bones-assets','Photoshop grid',asset_pattern),(ROOT/'f','Raw photo',raw_pattern)]:
    if not folder.is_dir():
        print(f'Optional reference folder not found: {folder}')
        continue
    for path in sorted(folder.rglob('*'),key=lambda p:str(p).lower()):
        if path.suffix.lower() not in {'.png','.jpg','.jpeg','.tif','.tiff'}:continue
        match=pattern.match(path.stem)
        if match is None:
            unparsed.append({'filename':path.name,'folder':folder.name});continue
        reference_rows.append({'individual_id':match['id'].replace(',','.').replace('-','.'),
            'reference_type':kind,'explicit_side':(match.groupdict().get('side') or '').upper(),
            'filename':path.name,'relative_path':path.relative_to(ROOT).as_posix()})
reference_manifest=pd.DataFrame(reference_rows,columns=['individual_id','reference_type','explicit_side','filename','relative_path'])
reference_manifest['in_metadata']=reference_manifest.individual_id.isin(metadata.individual_id)
reference_manifest.to_csv(ROOT/'reference_image_manifest.csv',index=False)
pd.DataFrame(unparsed,columns=['filename','folder']).to_csv(OUTPUT/'unparsed_reference_images.csv',index=False)
reference_lookup={}
for row in reference_rows:reference_lookup.setdefault(row['individual_id'],[]).append(row)
# Link references in an exported COPY; never overwrite the authoritative database.
linked_metadata=metadata.copy()
for kind,column in [('Photoshop grid','photoshop_reference_paths_json'),('Raw photo','raw_photo_paths_json')]:
    linked_metadata[column]=linked_metadata.individual_id.map(lambda ident:json.dumps([r['relative_path'] for r in reference_lookup.get(ident,[]) if r['reference_type']==kind]))
linked_metadata.to_csv(ROOT/'consolidated_metadata_with_references.csv',index=False)
metadata=linked_metadata
print(f'{len(reference_manifest)} references discovered; metadata copy written with links.')
display(reference_manifest.groupby('reference_type').size().rename('count'))

photo_heading=widgets.HTML()
photo_picker=widgets.Dropdown(description='Reference:',options=[],layout=widgets.Layout(width='100%'),style={'description_width':'initial'})
photo_previous=widgets.Button(description='Previous reference',layout=widgets.Layout(width='49%'))
photo_next=widgets.Button(description='Next reference',layout=widgets.Layout(width='49%'))
photo_size=widgets.IntSlider(value=900,min=400,max=1800,step=100,description='Preview pixels:',continuous_update=False,style={'description_width':'initial'},layout=widgets.Layout(width='95%'))
photo_image=widgets.Image(format='png',layout=widgets.Layout(width='100%',height='auto'))
photo_caption=widgets.HTML()
photo_state={'loading':False,'records':{}}

def current_editor_side():
    data=globals().get('editor_state',{}).get('data')
    return str(data['row'].get('side','')).upper() if data is not None else ''

def reference_priority(record,side):
    if record['reference_type']=='Raw photo':rank=3
    elif side and record['explicit_side']==side:rank=0
    elif not record['explicit_side']:rank=1
    else:rank=2
    return rank,record['filename'].lower()

def display_reference_photo(change=None):
    if photo_state['loading']:return
    values=[v for label,v in photo_picker.options]
    if not values or photo_picker.value is None:
        photo_image.value=b'';photo_caption.value='No linked reference image.'
        photo_previous.disabled=True;photo_next.disabled=True;return
    selected=photo_picker.value;index=values.index(selected);record=photo_state['records'][selected]
    photo_previous.disabled=index==0;photo_next.disabled=index==len(values)-1
    try:
        with Image.open(ROOT/selected) as source:
            rgba=ImageOps.exif_transpose(source).convert('RGBA')
            original_size=rgba.size
            preview=Image.alpha_composite(Image.new('RGBA',rgba.size,'white'),rgba).convert('RGB')
            preview.thumbnail((photo_size.value,photo_size.value),Image.Resampling.LANCZOS)
            buffer=io.BytesIO();preview.save(buffer,format='PNG')
        photo_image.value=buffer.getvalue()
        notice=''
        if record['explicit_side'] and current_editor_side() and record['explicit_side']!=current_editor_side():
            notice='<div><b>Opposite-side reference</b>: check anatomy; this reference is not mirrored.</div>'
        photo_caption.value=(f"<div><b>{html.escape(record['reference_type'])}</b> | {html.escape(record['filename'])} | {index+1}/{len(values)} | original {original_size[0]} × {original_size[1]} px</div>"+notice+'<div>Reference only: not registered to the mask. Place landmarks in the editor, not on this photo.</div>')
    except Exception as exc:
        photo_image.value=b'';photo_caption.value=html.escape(f'Could not display image: {exc}')

def load_reference_photos(ident):
    records=sorted(reference_lookup.get(str(ident),[]),key=lambda r:reference_priority(r,current_editor_side()))
    photo_state['loading']=True
    try:
        photo_state['records']={r['relative_path']:r for r in records}
        photo_heading.value=f'<h3>References — individual {html.escape(str(ident))}</h3>'
        photo_picker.options=[(f"{r['reference_type']} | {r['filename']}"+(f" | side {r['explicit_side']}" if r['explicit_side'] else ''),r['relative_path']) for r in records]
        photo_picker.value=records[0]['relative_path'] if records else None
    finally:photo_state['loading']=False
    display_reference_photo()

def step_reference_photo(offset):
    values=[v for label,v in photo_picker.options]
    if photo_picker.value not in values:return
    target=values.index(photo_picker.value)+offset
    if 0<=target<len(values):photo_picker.value=values[target]

photo_picker.observe(display_reference_photo,names='value')
photo_size.observe(display_reference_photo,names='value')
photo_previous.on_click(lambda _:step_reference_photo(-1))
photo_next.on_click(lambda _:step_reference_photo(1))
reference_panel=widgets.VBox([photo_heading,photo_picker,widgets.HBox([photo_previous,photo_next]),photo_size,photo_image,photo_caption],layout=widgets.Layout(width='48%',min_width='350px'))


126 references discovered; metadata copy written with links.


reference_type
Photoshop grid    27
Raw photo         99
Name: count, dtype: int64

## Review editor — click landmarks
Choose a point and click its location in the left image. Photoshop and raw references appear alongside. Turn toolbar pan/zoom off before placement. Save approved annotations before navigating away or restarting. Drafts persist during navigation but not kernel restart.

In [7]:
import copy
import hashlib
import html
import json
import uuid

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


# Close the figure from an earlier execution of this editor.
previous_editor_state = globals().get("editor_state", {})
if previous_editor_state.get("fig") is not None:
    plt.close(previous_editor_state["fig"])


# ------------------------------------------------------------
# Configuration and controls
# ------------------------------------------------------------

ids = sorted(working, key=float)
assert ids, "No working proposals available."

LANDMARKS = ["alpha", "beta", "gamma", "delta", "epsilon"]

for folder in ["reviewed_overlays", "annotations"]:
    (OUTPUT / folder).mkdir(parents=True, exist_ok=True)

picker = widgets.Dropdown(
    options=ids,
    description="Individual",
    layout=widgets.Layout(width="320px"),
)

previous_btn = widgets.Button(
    layout=widgets.Layout(width="49%"),
)

next_btn = widgets.Button(
    layout=widgets.Layout(width="49%"),
)

landmark_picker = widgets.ToggleButtons(
    options=LANDMARKS,
    value="alpha",
    description="Place:",
)

auto_advance = widgets.Checkbox(
    value=True,
    description="Advance to next landmark after clicking",
    indent=False,
)

angle = widgets.FloatText(
    value=0,
    description="Rotation °",
)

rotate_btn = widgets.Button(
    description="Apply rotation / reset proposals",
    layout=widgets.Layout(width="280px"),
)

preview_btn = widgets.Button(
    description="Refresh grid",
)

save_btn = widgets.Button(
    description="Save approved grid",
    button_style="success",
    layout=widgets.Layout(width="210px"),
)

approval = widgets.Checkbox(
    value=False,
    description="Anatomy and orientation verified",
    indent=False,
    layout=widgets.Layout(width="400px"),
)

notes = widgets.Textarea(
    description="Notes",
    layout=widgets.Layout(width="95%", height="80px"),
)

heading = widgets.HTML()
instruction = widgets.HTML()
view = widgets.Output()
ratios_view = widgets.Output()
message = widgets.Output()

editor_state = {
    "id": None,
    "data": None,
    "fig": None,
    "ax": None,
    "loading": False,
}

# In-memory drafts persist during navigation, not kernel restart.
editor_drafts = {}

# Used only to label saves made during this editor session.
saved_this_session = set()


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def notify(text):
    with message:
        clear_output(wait=True)
        print(text)


def filename_for(ident):
    row = working[ident]["row"]
    return str(row.get("filename", Path(row["path"]).name))


def destination_label(ident, direction):
    return (
        f"{direction}: {ident} | {filename_for(ident)}"
    )


def update_navigation():
    index = ids.index(picker.value)

    previous_btn.disabled = index == 0
    next_btn.disabled = index == len(ids) - 1

    previous_btn.description = (
        destination_label(ids[index - 1], "Previous")
        if index > 0 else
        "Previous: start of list"
    )

    next_btn.description = (
        destination_label(ids[index + 1], "Next")
        if index < len(ids) - 1 else
        "Next: end of list"
    )

    # Tooltips retain full names if button text is clipped.
    previous_btn.tooltip = previous_btn.description
    next_btn.tooltip = next_btn.description


def update_heading():
    ident = editor_state["id"]
    row = editor_state["data"]["row"]
    filename = str(row.get("filename", Path(row["path"]).name))
    saved = (
        " | saved this session"
        if ident in saved_this_session else
        ""
    )

    heading.value = (
        f"<h3>Individual {html.escape(str(ident))} "
        f"— {html.escape(filename)}</h3>"
        f"<div>Source side: {html.escape(str(row['side']))} "
        f"| {ids.index(ident) + 1} of {len(ids)}"
        f"{saved}</div>"
    )


def update_instruction(*_):
    instruction.value = (
        f"Select a landmark, then left-click its location. "
        f"Currently placing: <b>{landmark_picker.value}</b>. "
        f"Turn off the plot toolbar's pan/zoom tool before placing points."
    )


def cache_current():
    """Keep current point edits and notes when navigating."""
    if editor_state["id"] is None:
        return

    editor_drafts[editor_state["id"]] = {
        "data": copy.deepcopy(editor_state["data"]),
        "notes": notes.value,
        "approved": approval.value,
    }


def points_from_editor():
    return {
        name: [
            float(editor_state["data"]["points"][name][0]),
            float(editor_state["data"]["points"][name][1]),
        ]
        for name in LANDMARKS
    }


# ------------------------------------------------------------
# Plot and mouse interaction
# ------------------------------------------------------------

def preview(_=None):
    old_fig = editor_state.get("fig")

    if old_fig is not None:
        plt.close(old_fig)

    data = editor_state["data"]
    points = points_from_editor()
    ident = editor_state["id"]

    with view:
        clear_output(wait=True)

        # Prevent an extra automatically displayed figure.
        with plt.ioff():
            fig = plot_grid(
                data["mask"],
                points,
                f"{ident} | source {data['row']['side']} | review draft",
            )

        ax = fig.axes[0]
        ax.axis("on")

        # Explicit marker labels ensure all five points are visible.
        for name, (x, y) in points.items():
            selected = name == landmark_picker.value

            ax.scatter(
                [x], [y],
                s=85 if selected else 40,
                c="yellow" if selected else "red",
                edgecolors="black",
                zorder=20,
            )

            ax.annotate(
                name,
                (x, y),
                xytext=(7, 7),
                textcoords="offset points",
                fontsize=9,
                color="black",
                bbox={
                    "facecolor": "white",
                    "alpha": 0.8,
                    "edgecolor": "none",
                },
                zorder=21,
            )

        editor_state["fig"] = fig
        editor_state["ax"] = ax

        fig.canvas.mpl_connect(
            "button_press_event",
            on_image_click,
        )

        display(fig.canvas)

    with ratios_view:
        clear_output(wait=True)

        try:
            display(
                pd.Series(
                    geometry(points)[2],
                    name="Current ratios",
                )
            )
        except Exception as exc:
            print(
                "Grid geometry is not valid yet. "
                "Adjust the landmarks before saving."
            )
            print(f"{type(exc).__name__}: {exc}")

    update_heading()
    update_instruction()


def on_image_click(event):
    ax = editor_state.get("ax")

    if event.inaxes is not ax:
        return

    if event.button != 1:
        return

    if event.xdata is None or event.ydata is None:
        return

    # Do not move landmarks while using toolbar pan/zoom.
    toolbar = getattr(event.canvas, "toolbar", None)
    if toolbar is not None and getattr(toolbar, "mode", ""):
        return

    x = float(event.xdata)
    y = float(event.ydata)

    height, width = editor_state["data"]["mask"].shape

    if not (0 <= x < width and 0 <= y < height):
        return

    name = landmark_picker.value

    editor_state["data"]["points"][name] = [x, y]

    # A changed annotation requires verification again.
    approval.value = False

    if auto_advance.value:
        index = LANDMARKS.index(name)

        # Stop at epsilon rather than silently looping to alpha.
        if index < len(LANDMARKS) - 1:
            landmark_picker.value = LANDMARKS[index + 1]

    cache_current()
    preview()

    notify(
        f"Placed {name} at x={x:.1f}, y={y:.1f}. "
        "Changes are a draft until saved."
    )


# ------------------------------------------------------------
# Loading and navigation
# ------------------------------------------------------------

def load_individual(change=None):
    if editor_state["loading"]:
        return

    cache_current()
    ident = picker.value

    editor_state["loading"] = True

    try:
        if ident in editor_drafts:
            draft = editor_drafts[ident]

            editor_state["data"] = copy.deepcopy(draft["data"])
            notes.value = draft["notes"]
            approval.value = draft["approved"]
        else:
            editor_state["data"] = copy.deepcopy(working[ident])
            notes.value = ""
            approval.value = False

        editor_state["id"] = ident

        angle.value = float(
            editor_state["data"]["info"]["rotation_degrees"]
        )

        landmark_picker.value = "alpha"

    finally:
        editor_state["loading"] = False

    update_navigation()
    load_reference_photos(ident)
    preview()

    notify(
        "Draft loaded. Select a landmark and click the image. "
        "Navigation retains edits in memory; Save writes them to disk."
    )


def previous_individual(_):
    index = ids.index(picker.value)

    if index > 0:
        picker.value = ids[index - 1]


def next_individual(_):
    index = ids.index(picker.value)

    if index < len(ids) - 1:
        picker.value = ids[index + 1]


# ------------------------------------------------------------
# Rotation
# ------------------------------------------------------------

def apply_rotation(_):
    try:
        data = editor_state["data"]
        row = data["row"]

        mask, contour, info = load_working(
            row["path"],
            row["side"],
            angle.value,
        )

        points, fallback = propose(contour)

        editor_state["data"] = {
            "mask": mask,
            "points": points,
            "info": info,
            "row": row,
            "fallback": fallback,
        }

        approval.value = False
        landmark_picker.value = "alpha"

        cache_current()
        preview()

        notify(
            "Rotation applied. Landmarks were reset to new "
            "automatic proposals—review all five again."
        )

    except Exception as exc:
        notify(
            f"Rotation failed: {type(exc).__name__}: {exc}"
        )


# ------------------------------------------------------------
# Save approved annotation
# ------------------------------------------------------------

def save(_):
    try:
        if not approval.value:
            raise ValueError(
                "Check the anatomy/orientation verification box first."
            )

        if not OBSERVER_ID or not SESSION_ID:
            raise ValueError(
                "Observer and session identifiers are required."
            )

        data = editor_state["data"]
        points = points_from_editor()

        for xy in points.values():
            if not np.isfinite(xy).all():
                raise ValueError("Landmarks must have finite coordinates.")

        grid, lengths, ratios = geometry(points)

        height, width = data["mask"].shape

        if not all(
            0 <= xy[0] < width and 0 <= xy[1] < height
            for xy in points.values()
        ):
            raise ValueError("Points outside working image.")

        row = data["row"]
        ident = editor_state["id"]
        revision = uuid.uuid4().hex

        record = {
            **data["info"],
            "individual_id": ident,
            "observer_id": OBSERVER_ID,
            "session_id": SESSION_ID,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "revision": revision,
            "source_side": row["side"],
            "source_filename": str(
                row.get("filename", Path(row["path"]).name)
            ),
            "selection_reason": row.get("selection_reason", ""),
            "source_sha256": hashlib.sha256(
                Path(row["path"]).read_bytes()
            ).hexdigest(),
            "status": "observer_approved",
            "notes": notes.value,
            "background_tolerance": BACKGROUND_TOLERANCE,
            "right_already_reflected_setting": RIGHT_ALREADY_REFLECTED,
            "working_shape": list(data["mask"].shape),
            "beta_proposal_fallback": bool(data["fallback"]),
            "landmarks": points,
            "grid": {
                key: np.asarray(value).tolist()
                for key, value in grid.items()
            },
            "lengths_px": lengths,
            "ratios": ratios,
        }

        # Convert NumPy numeric types for JSON serialization.
        def json_default(value):
            if isinstance(value, np.ndarray):
                return value.tolist()
            if isinstance(value, np.generic):
                return value.item()
            raise TypeError(f"Cannot serialize {type(value).__name__}")

        record["reference_image_viewed_at_save"] = photo_picker.value
        record["reference_images_available"] = [value for label, value in photo_picker.options]

        payload = json.dumps(
            record,
            indent=2,
            default=json_default,
            allow_nan=False,
        )

        stamp = f"{ident}_{revision}"

        with plt.ioff():
            fig = plot_grid(
                data["mask"],
                points,
                f"{ident} | {OBSERVER_ID}/{SESSION_ID} | approved",
            )

        try:
            fig.savefig(
                OUTPUT / "reviewed_overlays" / f"{stamp}.png",
                dpi=160,
            )
        finally:
            plt.close(fig)

        (OUTPUT / "annotations" / f"{stamp}.json").write_text(
            payload,
            encoding="utf-8",
        )

        cache_current()
        saved_this_session.add(ident)
        update_heading()

        notify(
            f"Saved approved grid for individual {ident}. "
            "Use Next when ready. Run the export cell after reviewing."
        )

    except Exception as exc:
        notify(
            f"Not saved: {type(exc).__name__}: {exc}"
        )


# ------------------------------------------------------------
# Connect controls and show editor
# ------------------------------------------------------------

picker.observe(load_individual, names="value")
previous_btn.on_click(previous_individual)
next_btn.on_click(next_individual)

landmark_picker.observe(
    update_instruction,
    names="value",
)

preview_btn.on_click(preview)
rotate_btn.on_click(apply_rotation)
save_btn.on_click(save)

point_editor_panel=widgets.VBox([
    widgets.HBox([angle,rotate_btn]),landmark_picker,auto_advance,instruction,
    view,ratios_view,notes,approval,widgets.HBox([preview_btn,save_btn]),message
],layout=widgets.Layout(width='50%',min_width='450px'))
display(widgets.VBox([
    picker,
    widgets.HBox([previous_btn,next_btn],layout=widgets.Layout(width='100%')),
    heading,
    widgets.Box([point_editor_panel,reference_panel],layout=widgets.Layout(
        display='flex',flex_flow='row wrap',align_items='flex-start',
        justify_content='space-between',width='100%'))
]))
load_individual()


## Export current observer/session reviewed measurements
Run whenever you want updated CSVs. Keeps one latest approved revision per individual/observer/session, all revision history stays on disk. All metadata and canine fields are joined unchanged. Review status includes individuals not yet annotated. Raw segment lengths are working pixels, ratios dimensionless. Right-derived measurements are identifiable and should be included in a side-sensitivity analysis rather than claimed as exact replication of the left-only original study.

In [ ]:
annotations=[]
for p in (OUTPUT/'annotations').glob('*.json'):
    rec=json.loads(p.read_text())
    if rec['observer_id']!=OBSERVER_ID or rec['session_id']!=SESSION_ID:continue
    flat={k:v for k,v in rec.items() if k not in {'landmarks','grid','lengths_px','ratios','working_shape'}}
    flat.update({f'{name}_{axis}':xy[j] for name,xy in rec['landmarks'].items() for j,axis in enumerate(['x','y'])})
    flat.update({f'{s}_px':v for s,v in rec['lengths_px'].items()});flat.update(rec['ratios'])
    annotations.append(flat)
if annotations:
    reviewed=pd.DataFrame(annotations).sort_values('timestamp_utc').drop_duplicates(['individual_id','observer_id','session_id'],keep='last')
    reviewed=reviewed.sort_values('individual_id',key=lambda s:pd.to_numeric(s))
    tag=re.sub(r'[^A-Za-z0-9_.-]','_',f'{OBSERVER_ID}_{SESSION_ID}')
    reviewed.to_csv(OUTPUT/f'luna_ratios_reviewed_{tag}.csv',index=False)
    joined=metadata.merge(reviewed,on='individual_id',how='left',validate='one_to_one',suffixes=('','_luna')).sort_values('individual_id',key=lambda s:pd.to_numeric(s))
    joined.to_csv(OUTPUT/f'luna_ratios_metadata_canines_{tag}.csv',index=False)
    status=selected[['individual_id','side','filename']].copy();status['approved_this_session']=status.individual_id.isin(reviewed.individual_id)
    status.to_csv(OUTPUT/f'review_progress_{tag}.csv',index=False)
    display(reviewed[['individual_id','source_side']+list(RATIO_PAIRS)])
    print(f'{len(reviewed)} reviewed of {len(selected)} selected individuals')
else:print('No approved annotations saved for current observer/session yet.')


## Validation notes
Repeat point placement blinded to prior annotation and biological metadata, with another observer and later session, to assess landmark/ratio reliability. The original paper used two experienced observers, repeated observations at least a week apart, and randomly reordered ilia. This scaffold does not calculate ICC automatically or claim reliable anatomical detection.

FI/CF and DE/AD were the paper's strongest univariate ratios, with section points 0.99 and 0.76. These are not hard biological boundaries and were developed in 34 individuals aged 7–18. No sex decisions are made here. Do not infer male/female from PCA sign or apply logistic output coding without verification.

Geometry audit: A=(gamma_x,delta_y); B=(beta_x,delta_y); C=(alpha_x,delta_y); D=(gamma_x,beta_y); E=beta; F=(alpha_x,beta_y); G=(gamma_x,epsilon_y); H=(beta_x,epsilon_y); I=(alpha_x,epsilon_y). All seven ratios use Euclidean grid-segment lengths. Anatomical point placement, not algebra, is the limiting assumption.